In [0]:
eh_namespace = dbutils.secrets.get("aqi-secrets", "eventhub-namespace")
eh_name = dbutils.secrets.get("aqi-secrets", "eventhub-name")
eh_policy_name = dbutils.secrets.get("aqi-secrets", "eventhub-listen-policy")
eh_policy_key = dbutils.secrets.get("aqi-secrets", "eventhub-listen-key")

print("Namespace loaded:", eh_namespace is not None)
print("Event Hub name loaded:", eh_name is not None)
print("Policy name loaded:", eh_policy_name is not None)
print("Policy key loaded:", eh_policy_key is not None)

Namespace loaded: True
Event Hub name loaded: True
Policy name loaded: True
Policy key loaded: True


In [0]:
eh_connection_string = (
    f"Endpoint=sb://{eh_namespace}.servicebus.windows.net/;"
    f"SharedAccessKeyName={eh_policy_name};"
    f"SharedAccessKey={eh_policy_key}"
)

eh_bootstrap_servers = f"{eh_namespace}.servicebus.windows.net:9093"

eh_sasl = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{eh_connection_string}";'
)

print("Bootstrap servers:", eh_bootstrap_servers)
print("Event Hub:", eh_name)
print("SASL config created:", eh_sasl is not None)

Bootstrap servers: [REDACTED].servicebus.windows.net:9093
Event Hub: [REDACTED]
SASL config created: True


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AQI").getOrCreate()

raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", eh_bootstrap_servers)
    .option("subscribe", eh_name)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", eh_sasl)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
json_stream_df = raw_stream_df.selectExpr("CAST(value AS STRING) AS json_string")

In [0]:
checkpoint_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "checkpoints/eventhub_memory_test_clean_v3/"
)

query = (
    json_stream_df.writeStream
    .format("memory")
    .queryName("eventhub_json_test_clean")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.sql("""
SELECT *
FROM eventhub_json_test_clean
LIMIT 20
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_string                                                                                                                                                                                                                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"source": "test", "message": "hello event hubs"}                 

In [0]:
bronze_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "bronze/air_quality_raw/"
)

checkpoint_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "checkpoints/eventhub_to_bronze_v1/"
)

query = (
    json_stream_df.writeStream
    .format("parquet")
    .option("path", bronze_path)
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
spark.read.parquet(bronze_path).show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|json_string                                                                                                                                                                                                                                                                                                         |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"source": "test", "message": "hello event hubs"}                 

In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

aqi_schema = StructType(
    [
        StructField("source", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType()),
        StructField("city_id", IntegerType()),
        StructField("city", StringType()),
        StructField("country", StringType()),
        StructField("measurement_timestamp", StringType()),
        StructField("european_aqi", IntegerType()),
        StructField("pm10", DoubleType()),
        StructField("pm2_5", DoubleType()),
        StructField("nitrogen_dioxide", DoubleType()),
        StructField("ingestion_timestamp_utc", StringType())
        ])

bronze_raw_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "bronze/air_quality_raw/"
)

bronze_raw_df = spark.read.parquet(bronze_raw_path)


parsed_df = bronze_raw_df.select(
    from_json(col("json_string"), aqi_schema).alias("data")
)

final_df = parsed_df.select("data.*").filter(col("source") == "openmeteo")

In [0]:
final_df.show(truncate = False)

+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|source   |latitude|longitude|city_id|city        |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc         |
+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T00:00     |30          |22.7|15.3 |42.1            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T01:00     |30          |23.5|16.7 |44.3            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T04:00     |31          |26.6|16.6 |53.9            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-

In [0]:
bronze_structured_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "bronze/air_quality_structured/"
)

final_df.write.mode("overwrite").parquet(bronze_structured_path)

In [0]:
spark.read.parquet(bronze_structured_path).show(20, truncate=False)

+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|source   |latitude|longitude|city_id|city        |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc         |
+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T00:00     |30          |22.7|15.3 |42.1            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T01:00     |30          |23.5|16.7 |44.3            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09T04:00     |31          |26.6|16.6 |53.9            |2026-07-09T11:27:23.032781+00:00|
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-